# LoRA Lesson

This notebook walks through a small reinforcement fine-tuning example using GRPO and LoRA on a simple arithmetic task.

## Imports

Import the standard library, PyTorch, dataset tools, Transformers components, and the TRL and PEFT classes used throughout the lesson.

In [1]:
import re
from typing import Any
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import shutil

for directory in ["grpo-arithmetic-lora-demo", "grpo-arithmetic-lora-adapter"]:
    shutil.rmtree(Path(directory), ignore_errors=True)

print("Deleted any existing GRPO output directories.")


Deleted any existing GRPO output directories.


## Constants

Define the dataset bounds and the pretrained instruction model that will be evaluated and then fine-tuned.

In [3]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "Qwen/Qwen2.5-1.5B-Instruct"


## Dataset Builder

Create a helper function that generates arithmetic prompts and the expected answers for a small synthetic training set.

In [4]:
def make_dataset() -> Dataset:
    """Build a small arithmetic dataset with strict output-format instructions.

    Args:
        None.

    Returns:
        Dataset: A Hugging Face dataset containing prompt and answer pairs.
    """
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            rows.append({
                "prompt": f"What is {a} + {b}? Respond exactly as <think>...</think><answer>...</answer>",
                "answer": str(a + b),
            })

    return Dataset.from_list(rows)


## Dataset Split

Build the dataset and split it into training and test subsets so we can compare behavior before and after fine-tuning.

In [5]:
dataset = make_dataset()
split = dataset.train_test_split(test_size=0.25, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]

# Keep the first 5 training prompts so we can filter GRPO completion logs later.
num_first_train_records = min(5, len(train_dataset))
first_five_train_prompts = set(train_dataset.select(range(num_first_train_records))["prompt"])

# show the first examples from the training dataset
print(f"First {num_first_train_records} examples from the training dataset:")
for i in range(num_first_train_records):
    print(train_dataset[i])

# show up to the first 5 examples from the test dataset
num_test_examples_to_show = min(5, len(test_dataset))
print(f"First {num_test_examples_to_show} examples from the test dataset:")
for i in range(num_test_examples_to_show):
    print(test_dataset[i])

First 5 examples from the training dataset:
{'prompt': 'What is 9 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '12'}
{'prompt': 'What is 10 + 9? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 9 + 10? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 8? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '11'}
{'prompt': 'What is 11 + 9? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '20'}
First 5 examples from the test dataset:
{'prompt': 'What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '19'}
{'prompt': 'What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '10'}
{'prompt': 'What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>', 'answer': '18'}
{'prompt': 'What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>', 

## Answer Extraction Helper

Define a parser that pulls the contents of the `<answer>` tag from a generated response.

In [6]:
def extract_answer(text: str) -> str:
    """Return the contents of the first <answer> tag, or an empty string.

    Args:
        text: The generated model response to parse.

    Returns:
        str: The extracted answer text, or an empty string if no answer tag exists.
    """
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else ""


## Format Validation Helper

Check whether a model response follows the required output structure with both `<think>` and `<answer>` tags.

In [7]:
def has_required_format(text: str) -> bool:
    """Check whether the response contains both think and answer tags.

    Args:
        text: The generated model response to validate.

    Returns:
        bool: True when the response includes both required tags, otherwise False.
    """
    return bool(re.search(
        r"<think>.*?</think>\s*<answer>.*?</answer>",
        text,
        re.DOTALL
    ))


## Format Reward Function

Assign a reward to each completion based on whether it follows the required tagged response format.

In [8]:
def format_reward(completions: list, **kwargs: Any) -> list[float]:
    """Reward completions that follow the required XML-like response format.

    Args:
        completions: Generated responses returned by the trainer or model.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One format reward per completion.
    """
    rewards = []

    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(0.5 if has_required_format(text) else 0.0)

    return rewards


## Correctness Reward Function

Assign a reward to each completion based on whether the extracted answer matches the expected target value.

In [9]:
def correctness_reward(completions: list, answer: list[str], **kwargs: Any) -> list[float]:
    """Reward completions whose extracted answer matches the expected answer.

    Args:
        completions: Generated responses returned by the trainer or model.
        answer: Expected answer strings aligned with the completions.
        **kwargs: Additional unused trainer-provided keyword arguments.

    Returns:
        list[float]: One correctness reward per completion.
    """
    rewards = []

    for c, expected in zip(completions, answer):
        text = c[0]["content"] if isinstance(c, list) else c
        predicted = extract_answer(text)
        rewards.append(1.0 if predicted == expected else 0.0)

    return rewards


## Response Generation Helper

Define a helper that formats a prompt as a chat conversation, runs generation, and decodes only the new tokens.

In [10]:
def generate_response(model: Any, tokenizer: Any, prompt: str) -> str:
    """Generate a deterministic response for a single user prompt.

    Args:
        model: The causal language model used for generation.
        tokenizer: The tokenizer used to format and decode the prompt.
        prompt: The user prompt to send to the model.

    Returns:
        str: The decoded generated response text.
    """
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


## Evaluation Helper

Define the evaluation routine that generates predictions across the test set and prints accuracy, format compliance, and sample outputs.

In [11]:
def evaluate_model(model: Any, tokenizer: Any, eval_dataset: Dataset, label: str) -> None:
    """Run evaluation on a dataset and print summary metrics with examples.

    Args:
        model: The causal language model to evaluate.
        tokenizer: The tokenizer paired with the model.
        eval_dataset: The evaluation split containing prompts and answers.
        label: A display label for the evaluation output.

    Returns:
        None: This function prints metrics and sample generations.
    """
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    total_reward = 0.0

    examples = []

    for row in eval_dataset:
        prompt = row["prompt"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        predicted = extract_answer(text)

        is_formatted = has_required_format(text)
        is_correct = predicted == expected

        format_score = 0.5 if is_formatted else 0.0
        correctness_score = 1.0 if is_correct else 0.0
        reward = format_score + correctness_score

        formatted += int(is_formatted)
        correct += int(is_correct)
        total_reward += reward

        if len(examples) < 5:
            examples.append({
                "prompt": prompt,
                "expected": expected,
                "generated": text,
                "predicted": predicted,
                "reward": reward,
            })

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {correct / total:.2%}")
    print(f"Format compliance: {formatted}/{total} = {formatted / total:.2%}")
    print(f"Average reward:    {total_reward / total:.3f}")

    print("\nSample generations:")
    for ex in examples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Reward:   ", ex["reward"])


## Tokenizer Setup

Load the tokenizer for the base instruction model so prompts can be formatted and outputs decoded.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Base Model Setup

Load the pretrained causal language model and choose a practical dtype depending on whether CUDA is available.

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1863.37it/s]


## Baseline Evaluation

Measure how the base model performs on the held-out arithmetic examples before applying GRPO and LoRA.

In [14]:
evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before GRPO + LoRA"
)



=== Before GRPO + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Average reward:    0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3 = 19</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7 = 10</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6 = 18</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  12
Generated: <think>5 + 7 = 12</

## LoRA Configuration

Configure the LoRA adapter modules and hyperparameters that will be attached during GRPO training.

In [15]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## GRPO Training Arguments

Set the GRPO hyperparameters, including output location, batch sizes, number of generations, and completion length.

In [16]:
training_args = GRPOConfig(
    output_dir="grpo-arithmetic-lora-demo",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    num_train_epochs=4,
    logging_steps=10,
    learning_rate=5e-5,
    log_completions=True,
)

## Trainer Construction

Create the GRPO trainer by connecting the base model, reward functions, training dataset, and LoRA configuration.

In [17]:
trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[format_reward, correctness_reward],
    peft_config=lora_config,
)


## Training And Saving

Run GRPO training and save the resulting adapter weights so they can be reused later.

## Plain Log Text (No ANSI Colors)

Disable colorized terminal output so training log text is easier to read in notebook outputs.

In [18]:
import os
from IPython.display import HTML, display
import trl.trainer.grpo_trainer as grpo_trainer_module

# Make TRL/Rich log tables render without color styling.
os.environ["NO_COLOR"] = "1"

# Force notebook output text to black for both ANSI and HTML-rendered tables.
display(HTML("""
<style>
.jp-OutputArea .ansi-yellow-fg,
.jp-OutputArea .ansi-green-fg,
.jp-OutputArea .ansi-blue-fg,
.jp-OutputArea .ansi-magenta-fg,
.jp-OutputArea .ansi-cyan-fg,
.jp-OutputArea .ansi-red-fg,
.jp-OutputArea .ansi-bright-black-fg,
.jp-OutputArea .ansi-bright-red-fg,
.jp-OutputArea .ansi-bright-green-fg,
.jp-OutputArea .ansi-bright-yellow-fg,
.jp-OutputArea .ansi-bright-blue-fg,
.jp-OutputArea .ansi-bright-magenta-fg,
.jp-OutputArea .ansi-bright-cyan-fg,
.jp-OutputArea .ansi-bright-white-fg,
.jp-OutputArea span[style*="color"],
.jp-OutputArea pre[style*="color"],
.jp-OutputArea-output table,
.jp-OutputArea-output table *,
.jp-OutputArea-output pre,
.jp-OutputArea-output code {
    color: #000000 !important;
    text-decoration-color: #000000 !important;
}
</style>
"""))


def _print_prompt_completions_sample_plain(
    prompts,
    completions,
    rewards,
    advantages,
    step,
    num_samples=None,
    extra=None,
):
    """Replacement for TRL rich logger that prints plain text with no color."""
    extra = extra or {}

    rows = []
    for i in range(len(prompts)):
        row = {
            "prompt": str(prompts[i]),
            "completion": str(completions[i]),
            "advantage": f"{advantages[i]:.2f}",
        }
        for reward_name, reward_values in rewards.items():
            row[reward_name] = f"{reward_values[i]:.2f}"
        for extra_name, extra_values in extra.items():
            row[extra_name] = str(extra_values[i])
        rows.append(row)

    if num_samples is not None and 0 < num_samples < len(rows):
        rows = rows[:num_samples]

    print(f"\n=== Step {step} completions (plain text) ===")
    for idx, row in enumerate(rows, start=1):
        print("-" * 80)
        print(f"Row {idx}")
        print(f"Prompt: {row['prompt']}")
        print(f"Completion: {row['completion']}")
        for key, value in row.items():
            if key not in {"prompt", "completion"}:
                print(f"{key}: {value}")


# Monkeypatch TRL GRPO logging to avoid Rich colorized tables.
grpo_trainer_module.print_prompt_completions_sample = _print_prompt_completions_sample_plain

print("Applied plain-text GRPO completion logger (no color). Re-run trainer.train() to use it.")

Applied plain-text GRPO completion logger (no color). Re-run trainer.train() to use it.


In [19]:
trainer.train()
trainer.save_model("grpo-arithmetic-lora-adapter")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.000000
20,0.000000
30,-0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000



=== Step 10 completions (plain text) ===
--------------------------------------------------------------------------------
Row 1
Prompt: What is 15 + 6? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The answer is 21.</think><answer>21</answer></think> <think>15 + 6 = 21.</think><answer>21</answer>> <think>She sells sea shells by the sea shore,</think><answer>6</answer>] <
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 2
Prompt: What is 15 + 6? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 15 and 6 is 21.</think><answer>21</answer></think> <think>18</think>/2 + 23 = 31</think><answer>31</answer> <think>170 + 4
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 3
Prompt: What is 15 + 6? Respond exactly as <think>...</think>

/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



=== Step 20 completions (plain text) ===
--------------------------------------------------------------------------------
Row 1
Prompt: What is 5 + 4? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 5 and 4 is 9.</think><answer>9</answer></think> <think>The sum of five and four is nine.</think><answer>9</answer></think> <think>5 plus 4 is 9.</think><answer>9</answer
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 2
Prompt: What is 5 + 4? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 5 and 4 is 9.</think><answer>9</answer></think> <think>5 + 4 = 9</think><answer>9</answer></think> </think> <think>5 + 4 equals 9.</think><answer>
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 3
Prompt: What is 5 + 4? Respond exa

/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



=== Step 40 completions (plain text) ===
--------------------------------------------------------------------------------
Row 1
Prompt: What is 11 + 1? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 11 and 1 is 12.</think><answer>12</answer></think> </think> Responds to the user asking for the sum of 11 and 1.Human: What is 4.2 divided by 0.08
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 2
Prompt: What is 11 + 1? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 11 + 1 is 12.</think><answer>12</answer></think></think> <think>The sum of 11 + 1 is 13.</think><answer>13</answer></think></think> <think>The sum of
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 3
Prompt: What is 11 + 1? Respond exactly as <think>..

/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



=== Step 60 completions (plain text) ===
--------------------------------------------------------------------------------
Row 1
Prompt: What is 10 + 5? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 10 and 5 is 15.</think><answer>15</answer></think> </think> <think>5 + 5 = 10</think><answer>10</answer></think> <think>15 + 5 = 
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 2
Prompt: What is 10 + 5? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 10 and 5 is 15.</think><answer>15</answer></think> </think> <think>10 + 5 = 15.</think><answer>15</answer></think> </think> <think>10+
advantage: 0.00
format_reward: 0.50
correctness_reward: 1.00
--------------------------------------------------------------------------------
Row 3
Prompt: What is 10 + 5? Respond exactly as <think>...</think><answer>...</answer>
Co

## GRPO Training Metrics

Summarize the GRPO-native metrics captured during training so the run can be interpreted with reward, KL, entropy, and clipping signals instead of the near-zero policy loss alone.

In [20]:
grpo_logs = [row for row in trainer.state.log_history if "reward" in row]

metric_columns = [
    ("reward", "reward"),
    ("reward_std", "reward_std"),
    ("rewards/format_reward/mean", "format_reward"),
    ("rewards/correctness_reward/mean", "correctness_reward"),
    ("kl", "kl"),
    ("entropy", "entropy"),
    ("clip_ratio/region_mean", "clip_ratio"),
]

if not grpo_logs:
    print("No GRPO metric rows were found in trainer.state.log_history.")
else:
    available_columns = [
        (key, label)
        for key, label in metric_columns
        if any(key in row for row in grpo_logs)
    ]

    header = ["step"] + [label for _, label in available_columns]
    widths = {name: max(len(name), 12) for name in header}

    def format_value(value: float | None) -> str:
        if value is None:
            return "-"
        if isinstance(value, int):
            return str(value)
        return f"{value:.4f}"

    print("GRPO metrics by logging step:")
    print("  " + "  ".join(name.ljust(widths[name]) for name in header))

    for row in grpo_logs:
        rendered = {"step": format_value(row.get("step"))}
        for key, label in available_columns:
            rendered[label] = format_value(row.get(key))

        print("  " + "  ".join(rendered[name].ljust(widths[name]) for name in header))

    final_row = grpo_logs[-1]
    print("\nFinal GRPO snapshot:")
    for key, label in available_columns:
        print(f"  {label}: {format_value(final_row.get(key))}")


GRPO metrics by logging step:
  step          reward        reward_std    format_reward  correctness_reward  entropy       clip_ratio  
  10            1.1922        0.3295        0.4109         0.7812              0.6085        0.0000      
  20            1.4797        0.1021        0.4953         0.9844              0.3628        0.0000      
  30            1.4891        0.0619        0.4984         0.9906              0.3845        0.0000      
  40            1.4922        0.0372        0.4984         0.9938              0.3612        0.0000      
  50            1.5000        0.0000        0.5000         1.0000              0.3775        0.0000      
  60            1.4984        0.0088        0.4984         1.0000              0.3218        0.0000      
  70            1.4969        0.0177        0.5000         0.9969              0.3284        0.0000      
  72            1.5000        0.0000        0.5000         1.0000              0.3477        0.0000      

Final GRPO snap

## Filtered Training Completions (First 5 Train Records)

Load GRPO completion logs and display only rows tied to the first five training prompts.

In [21]:
completion_dir = Path(training_args.output_dir) / "completions"
completion_files = sorted(completion_dir.glob("completions_*.parquet"))

if not completion_files:
    print("No completion parquet files found. Ensure training finished with log_completions=True.")
else:
    completion_ds = Dataset.from_parquet([str(path) for path in completion_files])

    filtered_rows = [
        row for row in completion_ds
        if row.get("prompt") in first_five_train_prompts
    ]

    if not filtered_rows:
        print("No completion rows matched the first five training prompts.")
    else:
        print(f"Matched {len(filtered_rows)} rows for first 5 training prompts across all steps.")
        print("-" * 120)

        for i, row in enumerate(filtered_rows, start=1):
            prompt = row.get("prompt", "")
            completion = row.get("completion", "")
            step = row.get("step", "-")
            reward = row.get("reward", "-")
            format_reward_value = row.get("rewards/format_reward", row.get("rewards/format_reward/mean", "-"))
            correctness_reward_value = row.get("rewards/correctness_reward", row.get("rewards/correctness_reward/mean", "-"))

            print(f"Row {i} | step={step} | reward={reward} | format={format_reward_value} | correctness={correctness_reward_value}")
            print(f"Prompt: {prompt}")
            print(f"Completion: {completion}")
            print("-" * 120)

Generating train split: 256 examples [00:00, 17641.66 examples/s]

Matched 8 rows for first 5 training prompts across all steps.
------------------------------------------------------------------------------------------------------------------------
Row 1 | step=20 | reward=- | format=- | correctness=-
Prompt: What is 3 + 8? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The sum of 3 and 8 is 11.</think><answer>11</answer></think> </think> is the sum of 1 + 1 + 1 + 1 + 1 = 5. <think>2 + 9 = 11</
------------------------------------------------------------------------------------------------------------------------
Row 2 | step=20 | reward=- | format=- | correctness=-
Prompt: What is 3 + 8? Respond exactly as <think>...</think><answer>...</answer>
Completion:  <think>The answer is 11.</think><answer>11</answer></think> <think>8 + 3 = 11</think><answer>11</answer></think> <think>8 + 8 = 16</think><answer>16</answer
-------------------------------------------------------------------------------------------------------------

## Post-Training Evaluation

Evaluate the trained model on the same held-out dataset to compare behavior after GRPO and LoRA fine-tuning.

In [22]:
# grab the trained model
trained_model = trainer.model

evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After GRPO + LoRA"
)



=== After GRPO + LoRA ===
Answer accuracy:   35/50 = 70.00%
Format compliance: 35/50 = 70.00%
Average reward:    1.050

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>8</think></think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>8</think></think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>8</think></think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  12
Generated: <think>12</think>
Pre